# Second Models: Four-Model Retrieval Comparison

Run all cells with Internet access and a CUDA GPU. This notebook works on Kaggle, Colab, and local Jupyter using standard Python.

Compare frozen pretrained DINO ViT-S/16, supervised DeiT-S/16, DINOv2 ViT-B/14, and supervised DeiT-B/16 in one session. No training or fine-tuning is performed.

Outputs in `results/02_second_models/`:
- `metrics_validation.csv`
- `metrics_test.csv`
- `runtime_summary.csv`
- `run_metadata.json`

Two ZIP files are saved **next to the repository directory**:
- `second_models_results.zip`: four result files and the shared split manifest.
- `second_models_embeddings.zip`: embeddings for every model and split, cache metadata, run metadata, and the shared split manifest.

Rankings and failure-case figures are not saved by this notebook.

## 1. Source and experiment configuration

All four models use the same 1,840-image gallery, 1,840 validation queries, and 3,669 test queries across 37 breeds. Relevance means breed agreement. Use the agreed configuration unchanged; the test split has already been evaluated in earlier stages.

In [ ]:
from pathlib import Path
import datetime as dt
import hashlib
import json
import os
import shlex
import subprocess
import sys
import uuid

REPOSITORY_URL = "https://github.com/kite121/Visual_retrieval_system.git"
BRANCH = "main"
MODELS = [
    "dino_vit_s16",
    "deit_vit_s16_supervised",
    "dinov2_vit_b14",
    "deit_vit_b16_supervised",
]
BATCH_SIZE = 64
NUM_WORKERS = 2
TIMING_REPEATS = 3
EXPECTED_MANIFEST_SHA256 = "82f40e25888189aba46d1c1ff2c74894284c1ff7fa50362026c8abbea0ec9f80"

def run(command):
    command = [str(part) for part in command]
    print("$", shlex.join(command), flush=True)
    subprocess.run(command, cwd=REPO_DIR, check=True)

start_dir = Path.cwd().resolve()
local_repositories = [
    parent for parent in (start_dir, *start_dir.parents)
    if (parent / ".git").is_dir()
    and (parent / "scripts" / "run_evaluation.py").is_file()
]
if local_repositories:
    REPO_DIR = local_repositories[0]
else:
    REPO_DIR = start_dir / "Visual_retrieval_system"
    if not REPO_DIR.exists():
        subprocess.run([
            "git", "clone", "--branch", BRANCH, "--depth", "1",
            REPOSITORY_URL, str(REPO_DIR),
        ], check=True)
    else:
        if not (REPO_DIR / ".git").is_dir():
            raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout")
        run(["git", "checkout", BRANCH])
        run(["git", "pull", "--ff-only", "origin", BRANCH])

os.chdir(REPO_DIR)
SOURCE_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True,
).strip()
RUN_ID = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
RESULT_DIR = REPO_DIR / "results" / "02_second_models"
ARTIFACT_DIR = REPO_DIR / "artifacts" / "02_second_models" / RUN_ID
TIMING_DIR = ARTIFACT_DIR / "timing_measurements"
print("Repository:", REPO_DIR)
print("Source commit:", SOURCE_COMMIT)
print("Run ID:", RUN_ID)

## 2. Dependencies and GPU

Install the single root dependency file. Existing compatible PyTorch and torchvision installations are retained. The full four-model evaluation requires a CUDA GPU. A smaller batch size can be selected in the configuration cell before starting.

In [ ]:
run([
    sys.executable, "-m", "pip", "install",
    "--disable-pip-version-check", "-q", "-r", "requirements.txt",
])

probe = subprocess.check_output([
    sys.executable, "-c",
    "import json,sys,torch; sys.path.insert(0,'src'); "
    "from visual_retrieval import models,timing; "
    "assert torch.cuda.is_available(), 'Enable a CUDA GPU before running this notebook'; "
    "print(json.dumps({'hardware':timing.hardware_info('cuda'),"
    "'registered_models':models.available_models()}))",
], cwd=REPO_DIR, text=True)
environment = json.loads(probe)
assert set(MODELS).issubset(environment["registered_models"])
DEVICE = "cuda"
print(json.dumps(environment["hardware"], indent=2))

help_text = subprocess.check_output([
    sys.executable, "scripts/run_evaluation.py", "--help",
], cwd=REPO_DIR, text=True)
assert "--no-save-rankings" in help_text and "--experiment-id" in help_text, (
    "Update the repository checkout to the current main branch."
)

## 3. Download data and verify the shared split

Reconstruct the seed-42 split and compare it with the committed manifest. The manifest is not replaced. Both query splits must be disjoint from the gallery.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import FileLink, display

run([sys.executable, "scripts/prepare_dataset.py", "--check"])
manifest_path = REPO_DIR / "configs" / "split_manifest.csv"
manifest_bytes = manifest_path.read_bytes()
assert hashlib.sha256(manifest_bytes).hexdigest() == EXPECTED_MANIFEST_SHA256
manifest = pd.read_csv(manifest_path)
assert manifest["split"].value_counts().to_dict() == {
    "gallery": 1840, "val": 1840, "test": 3669,
}
assert manifest["image_id"].is_unique and manifest["breed"].nunique() == 37
display(manifest.groupby("split").agg(
    images=("image_id", "size"), breeds=("breed", "nunique"),
).reindex(["gallery", "val", "test"]))

## 4. Quality evaluation

Run all four models in one invocation so the canonical files contain all four rows. Extract fresh gallery, validation, and test embeddings; calculate Precision@5/10, Recall@5/10, and Hit@5. Similarity search is used to compute metrics, but ranking arrays are not written.

The initial single-run runtime table is replaced in the next step by the three-repeat summary.

In [ ]:
run([
    sys.executable, "scripts/run_evaluation.py",
    "--models", *MODELS,
    "--device", DEVICE,
    "--batch-size", BATCH_SIZE,
    "--num-workers", NUM_WORKERS,
    "--results-dir", RESULT_DIR,
    "--artifacts", ARTIFACT_DIR,
    "--experiment-id", "02_second_models",
    "--output-format", "canonical",
    "--no-save-rankings",
    "--rebuild",
])

quality_runs = [
    json.loads((ARTIFACT_DIR / "runs" / model / "evaluation.json").read_text())
    for model in MODELS
]
expected_timing_ids = quality_runs[0]["query_latency"]["query_ids"]
assert len(expected_timing_ids) == len(set(expected_timing_ids)) == 200
for result in quality_runs:
    assert result["pretrained"] and not result["gallery_build_cache_reused"]
    assert result["manifest_sha256"] == EXPECTED_MANIFEST_SHA256
    assert result["sizes"] == {"gallery": 1840, "val": 1840, "test": 3669}
    assert result["query_latency"]["query_ids"] == expected_timing_ids
    assert result["hardware"] == environment["hardware"]
    assert set(result["metrics"]) == {"val", "test"}
    for split in ("val", "test"):
        assert set(result["metrics"][split]) == {"P@5", "P@10", "R@5", "R@10", "Hit@5"}
        assert all(np.isfinite(v) and 0 <= v <= 1 for v in result["metrics"][split].values())
assert not list(ARTIFACT_DIR.rglob("topk_*.npz"))
print("Quality evaluation complete for all four models.")

## 5. Repeated runtime measurements

Measure the same 200 query IDs three times per model. Rebuild gallery features for each repeat, rotate model order, and use GPU synchronization and warm-up.

Only `runtime_summary.csv` is exported. Its latency columns contain the median, minimum, and maximum **of the three per-repeat medians**; gallery columns contain the median, minimum, and maximum of the three gallery build times. They are not confidence intervals or per-query extrema. Raw timing records remain local for validation.

In [ ]:
run([
    sys.executable, "scripts/benchmark_timing.py",
    "--models", *MODELS,
    "--device", DEVICE,
    "--batch-size", BATCH_SIZE,
    "--num-workers", NUM_WORKERS,
    "--repeats", TIMING_REPEATS,
    "--output-dir", TIMING_DIR,
    "--summary-dir", RESULT_DIR,
])

timing_records = []
for model in MODELS:
    for repeat in range(1, TIMING_REPEATS + 1):
        path = TIMING_DIR / f"{model}_repeat{repeat}.json"
        record = json.loads(path.read_text())
        assert record["model"] == model and record["repeat"] == repeat
        assert record["pretrained"] and not record["gallery_build_cache_reused"]
        assert record["manifest_sha256"] == EXPECTED_MANIFEST_SHA256
        assert record["query_latency"]["query_ids"] == expected_timing_ids
        assert record["hardware"] == environment["hardware"]
        assert record["batch_size"] == BATCH_SIZE
        assert record["num_workers"] == NUM_WORKERS
        timing_records.append(record)

runtime_summary = pd.read_csv(RESULT_DIR / "runtime_summary.csv")
assert set(runtime_summary["model_id"]) == set(MODELS)
assert runtime_summary["model_id"].is_unique
assert (runtime_summary["repeats"] == TIMING_REPEATS).all()
for model in MODELS:
    row = runtime_summary.set_index("model_id").loc[model]
    records = [r for r in timing_records if r["model"] == model]
    for kind in ("embed", "search", "preprocess"):
        values = [r["query_latency"][f"{kind}_ms"]["median"] for r in records]
        for label, operation in (("median", np.median), ("min", np.min), ("max", np.max)):
            assert np.isclose(row[f"{kind}_{label}_ms"], operation(values))
    values = [r["gallery_build_seconds"] for r in records]
    for label, operation in (("median", np.median), ("min", np.min), ("max", np.max)):
        assert np.isclose(row[f"gallery_build_seconds_{label}"], operation(values))
print("Verified 12 measurements and four summary rows.")

## 6. Complete run metadata and inspect results

Record the Git commit, actual installed package versions, precise timing aggregation, and the common latency-query sample. The four result files contain the same experiment identifier and hardware context.

In [ ]:
from importlib.metadata import version

metadata_path = RESULT_DIR / "run_metadata.json"
metadata = json.loads(metadata_path.read_text())
assert metadata["experiment_id"] == "02_second_models"
metadata["run_id"] = RUN_ID
metadata["source_commit"] = SOURCE_COMMIT
metadata["completed_utc"] = dt.datetime.now(dt.timezone.utc).isoformat()
metadata["environment"]["packages"] = {
    package: version(package)
    for package in (
        "torch", "torchvision", "timm", "numpy", "pandas",
        "scikit-learn", "Pillow", "PyYAML",
    )
}
metadata["timing"].update({
    "repeats_per_model": TIMING_REPEATS,
    "query_ids": expected_timing_ids,
    "model_order": "rotated between repeats",
    "runtime_summary_source": "three fresh measurements per model after quality evaluation",
    "latency_aggregation": "median/min/max of the per-repeat median latency",
    "gallery_aggregation": "median/min/max of the per-repeat gallery build time",
})
metadata["artifacts"] = {
    "rankings_saved": False,
    "embeddings_archive": "second_models_embeddings.zip",
    "embedding_splits": ["gallery", "val", "test"],
}
metadata_path.write_text(json.dumps(metadata, indent=2) + "\n")

validation_metrics = pd.read_csv(RESULT_DIR / "metrics_validation.csv")
test_metrics = pd.read_csv(RESULT_DIR / "metrics_test.csv")
for frame in (validation_metrics, test_metrics):
    assert frame["model_id"].is_unique and set(frame["model_id"]) == set(MODELS)
    values = frame.drop(columns=["model_id", "model"]).to_numpy()
    assert np.isfinite(values).all() and ((values >= 0) & (values <= 1)).all()

print("Validation metrics")
display(validation_metrics.round(4))
print("Test metrics")
display(test_metrics.round(4))
print("Repeated runtime summary")
display(runtime_summary.round(3))

## 7. Compare quality with the historical four-model experiment

Compare every quality value against the committed Stage 3 results. A difference is reported explicitly. The historical values remain unchanged, and timing differences are expected when hardware or environment changes.

In [ ]:
historical = pd.read_csv(REPO_DIR / "results" / "stage3" / "quality" / "metrics.csv")
historical = historical.set_index(["model", "split"])
metric_names = {
    "precision_at_5": "P@5", "recall_at_5": "R@5",
    "precision_at_10": "P@10", "recall_at_10": "R@10",
    "hit_at_5": "Hit@5",
}
comparison_rows = []
for split, frame in (("val", validation_metrics), ("test", test_metrics)):
    for row in frame.to_dict("records"):
        old = historical.loc[(row["model"], split)]
        for column, old_column in metric_names.items():
            comparison_rows.append({
                "model_id": row["model_id"], "split": split,
                "metric": column, "previous": float(old[old_column]),
                "current": float(row[column]),
                "absolute_difference": abs(float(row[column]) - float(old[old_column])),
            })
comparison = pd.DataFrame(comparison_rows)
assert len(comparison) == 40
max_difference = float(comparison["absolute_difference"].max())
print("Compared quality values:", len(comparison))
print("Maximum absolute difference:", max_difference)
if max_difference <= 1e-6:
    print("Quality metrics match the historical experiment within 1e-6.")
else:
    print("Quality differs: review these rows before replacing historical results.")
    display(comparison.loc[comparison["absolute_difference"] > 1e-6])
metadata["historical_comparison"] = {
    "source": "results/stage3/quality/metrics.csv",
    "values_compared": len(comparison),
    "max_absolute_difference": max_difference,
    "tolerance": 1e-6,
    "within_tolerance": max_difference <= 1e-6,
}
metadata_path.write_text(json.dumps(metadata, indent=2) + "\n")

## 8. Validate embeddings

Require complete image-ID and label coverage, finite L2-normalized vectors, and the expected feature dimension for every model and split. The four models produce 12 embedding arrays in total.

In [ ]:
dimension_by_model = {
    "dino_vit_s16": 384,
    "deit_vit_s16_supervised": 384,
    "dinov2_vit_b14": 768,
    "deit_vit_b16_supervised": 768,
}
embedding_files = []
for model in MODELS:
    for split in ("gallery", "val", "test"):
        expected = manifest.loc[manifest["split"] == split].sort_values("image_id", kind="stable")
        cache_dir = ARTIFACT_DIR / "embeddings" / model
        array_path = cache_dir / f"{split}.npz"
        info_path = cache_dir / f"{split}.json"
        with np.load(array_path, allow_pickle=False) as arrays:
            vectors = arrays["embeddings"]
            assert arrays["image_ids"].tolist() == expected["image_id"].tolist()
            assert arrays["labels"].tolist() == expected["label"].tolist()
            assert vectors.shape == (len(expected), dimension_by_model[model])
            assert np.isfinite(vectors).all()
            assert np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-3)
        cache_metadata = json.loads(info_path.read_text())
        assert cache_metadata["pretrained"] and not cache_metadata["cache_reused"]
        assert cache_metadata["n_images"] == len(expected)
        embedding_files.extend([array_path, info_path])
assert len(embedding_files) == 24
print("Verified 12 embedding arrays and 12 cache metadata files.")

## 9. Export two separate ZIP files

Both ZIP files are created in the parent directory of the repository. Embeddings are stored as feature arrays; pretrained model weight files and the dataset are not bundled. The ZIP contents are checked against the source files before download.

In [ ]:
import zipfile

expected_results = {
    "metrics_validation.csv", "metrics_test.csv",
    "runtime_summary.csv", "run_metadata.json",
}
assert {p.name for p in RESULT_DIR.iterdir()} == expected_results
results_archive = REPO_DIR.parent / "second_models_results.zip"
embeddings_archive = REPO_DIR.parent / "second_models_embeddings.zip"

def write_and_verify_archive(path, entries):
    checksums = {}
    with zipfile.ZipFile(path, "w", zipfile.ZIP_DEFLATED) as archive:
        for source, archive_name in entries:
            archive.write(source, archive_name)
            checksums[archive_name] = hashlib.sha256(source.read_bytes()).hexdigest()
    with zipfile.ZipFile(path) as archive:
        assert archive.testzip() is None
        assert set(archive.namelist()) == set(checksums)
        for archive_name, expected_hash in checksums.items():
            assert hashlib.sha256(archive.read(archive_name)).hexdigest() == expected_hash
    print(f"Created {path} ({path.stat().st_size / 1024**2:.1f} MiB)")
    display(FileLink(os.path.relpath(path, Path.cwd())))

common_entries = [
    (manifest_path, "configs/split_manifest.csv"),
]
write_and_verify_archive(results_archive, [
    *common_entries,
    *[(RESULT_DIR / name, f"results/02_second_models/{name}") for name in sorted(expected_results)],
])
write_and_verify_archive(embeddings_archive, [
    *common_entries,
    (metadata_path, "results/02_second_models/run_metadata.json"),
    *[(path, "artifacts/02_second_models/" + path.relative_to(ARTIFACT_DIR).as_posix())
      for path in embedding_files],
])

## Completion

Download both ZIP files before ending the session. The result ZIP is intended for review and Git publication. The embeddings ZIP can be stored separately and reused for the demo or a teammate's later analysis.

The notebook reports any differences from the historical quality results. Review them before changing report numbers. Failure-case selection and ranking export are handled separately by the teammate responsible for that work.